# Index spread moves

Pick an index (`HY47`, `HY42`, `XO44`, `IG45`, `MA40`, ...) and two dates. First it shows which names are missing data, then one row per index member with the spread, `upfront`, `all_upfront` and `risky_pv01` at both dates and the change in each, sorted by whichever change you pick (`SORT_BY`).

- Reads the published store `data/base` only (no Databricks / Bloomberg / Excel, and this notebook never writes data). The first cell prints which snapshot it is (published date and last price date). The data is built and published by the owner with the scripts in `pricing/`.
- Spread is `conv_spread` in bp at the index maturity; `upfront` and `all_upfront` are in points. Dates snap to the last available date on or before the request, and an index has no data before its launch date.
- Names with no price on a date stay in the table with blank values and a `status` saying why: `missing` (no Databricks row, red). Rows shaded amber are wide names (`recal`), blue rows are re-priced from the other coupon (`recoupon`), purple rows come from the Markit add-in (`addin`). A `spread_override` from `overrides.csv` beats all of these.

In [1]:
import sys
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'analysis' / 'cdx_spreads.py').exists())     # project root, wherever the notebook is opened from
sys.path.insert(0, str(ROOT / 'analysis'))

import pandas as pd
import cdx_spreads as cs
from IPython.display import HTML, display

pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 300)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.expand_frame_repr', False)

# no-wrap HTML table with fixed decimals, blanks for missing, red rows for names missing data
show = cs.show_table


print(', '.join(cs.load_index_list()['index_name']))
cs.print_data_as_of()          # which published snapshot (data/) this notebook is reading


IG47, IG46, IG45, IG44, IG43, IG42, IG41, IG40, IG39, IG38, HY47, HY46, HY45, HY44, HY43, HY42, HY41, HY40, HY39, HY38, XO46, XO45, XO44, XO43, XO42, XO41, XO40, XO39, XO38, XO37, MA46, MA45, MA44, MA43, MA42, MA41, MA40, MA39, MA38, MA37
data published 2026-10-27 11:10; prices through 2026-10-26


## Parameters

In [2]:
INDEX = 'XO46'
DATE_X = '2026-09-29'   # start
DATE_Y = '2026-10-26'   # end
SORT_BY = 'spread'      # which change to sort on: 'spread', 'upfront', 'all_upfront' or 'risky_pv01'
SORT_BY_ABS = False     # False: largest positive change on top (widest first for spreads). True: largest move in either direction first.

## 1. Names not priced
**Priced** means a curve point has a `par_spread`, or a `conv_spread` / `upfront` together with the coupon it is quoted at (`risky_pv01` is optional: `cds_pricer.py` derives it from a flat curve). A `conv_spread` of 0 is Markit's 'no spread solves' placeholder, not a price. The goal is for every name on every date to be priced.

Sources, in order: **Databricks** first, then the **Markit add-in** (`addin_fill.py`) for cells Databricks could not price, then any manual override. Every spread, upfront, `all_upfront` and `risky_pv01` below is **at the index coupon** (500 for HY/XO, 100 for IG/MA).

- **Red box:** names with no price on one of the dates, grouped by reason (*never priced in this index*, *conv=0*, *incomplete* stub row, *gap*).
- **Amber box:** wide names priced with a marked-down recovery (`recovery = 100 - all_upfront - 5`, flat 4% discounting): a clean comparable spread, but not Markit's.
- **Blue box:** names Markit quotes at the other coupon, re-priced to the index coupon with `cds_pricer.py`.
- **Purple box:** names priced from the Markit add-in because Databricks has no price (renamed or missing curves); half-year tenors are interpolated between annual nodes.

The table gives, for each flagged name, `days_with_data` of `days_in_index` (priced by either source), `first_data`, the coupon it was quoted at (for add-in and override rows, the index coupon they were priced at) and the recovery used on each date (40% unless marked down).

In [3]:
res = cs.spread_changes(INDEX, DATE_X, DATE_Y)
a = res.attrs
own, recoupon, recal, addin, manual, unpriced = cs.priced_counts(res)
print(f"{INDEX}: conv_spread {a['date_x']:%Y-%m-%d} -> {a['date_y']:%Y-%m-%d} | index maturity {a['maturity']:%Y-%m-%d} | index coupon {a['index_coupon']} | "
      f"{len(res)} names: {own} priced by Databricks at the index coupon, {recoupon} re-priced from the other coupon, {recal} wide names at a marked-down recovery, "
      f"{addin} priced from the Markit add-in, {manual} manual overrides, {unpriced} not priced")
display(HTML(cs.missing_banner_html(res)))
show(cs.missing_report(res), numeric={'quoted coupon X': 0, 'quoted coupon Y': 0, 'recovery X (%)': 1, 'recovery Y (%)': 1})

XO46: conv_spread 2026-09-29 -> 2026-10-26 | index maturity 2031-12-20 | index coupon 500 | 75 names: 70 priced by Databricks at the index coupon, 2 re-priced from the other coupon, 1 wide names at a marked-down recovery, 2 priced from the Markit add-in, 0 manual overrides, 0 not priced


member_name,Market_CDS_Ticker,what,status,days_with_data,days_in_index,first_data,quoted coupon X,quoted coupon Y,recovery X (%),recovery Y (%)
Virgin Media Finance PLC,VMED,"Priced, wide name: conv_spread at a marked-down recovery",Y recal,26,26,2026-09-21,500,500,40.0,20.1
SES SA,SESGFP,"Priced, re-priced from the other coupon to the index coupon",X recoupon; Y recoupon,26,26,2026-09-21,100,100,40.0,40.0
Suedzucker AG,SZUGR,"Priced, re-priced from the other coupon to the index coupon",X recoupon; Y recoupon,26,26,2026-09-21,100,100,40.0,40.0
Metlen Energy & Metals SA,MYTIL,Priced from the Markit add-in (Databricks has no price),X addin; Y addin,26,26,2026-09-21,500,500,40.0,40.0
Motion Finco Sarl,MERLLN B,Priced from the Markit add-in (Databricks has no price),X addin; Y addin,26,26,2026-09-21,500,500,40.0,40.0


## 2. Largest moves
Sorted by the change chosen in `SORT_BY` (see `SORT_BY_ABS` above). Row color shows where a name's numbers came from: plain rows are white on black, tinted rows are black on a light tint.

| Row shading | Meaning |
|---|---|
| black (no tint) | priced by Databricks at the index coupon |
| blue | Markit quotes the name at the other coupon; re-priced to the index coupon with `cds_pricer` |
| purple | Databricks has no price on at least one date; taken from the Markit add-in |
| amber, bold | wide name: no conventional spread solves at 40% recovery, so it is priced at a marked-down recovery (recovery columns show which) |
| red, bold | no price on at least one of the two dates; blank spreads, sorted to the bottom |

In [4]:
print(f"median spread change {res['change'].median():.1f} bp | median pct change {res['pct_change'].median():.1f}%")

sort_col = {'spread': 'change', 'upfront': 'upfront_change', 'all_upfront': 'all_upfront_change', 'risky_pv01': 'risky_pv01_change'}[SORT_BY]
res = res.sort_values(sort_col, key=abs if SORT_BY_ABS else None, ascending=False, na_position='last').reset_index(drop=True)

dx, dy = f"{a['date_x']:%Y-%m-%d}", f"{a['date_y']:%Y-%m-%d}"
cols = {'conv_spread_x': f'spread {dx}', 'conv_spread_y': f'spread {dy}', 'change': 'spread change (bp)', 'pct_change': 'spread change (%)'}
for m in ('upfront', 'all_upfront', 'risky_pv01'):
    cols |= {f'{m}_x': f'{m} {dx}', f'{m}_y': f'{m} {dy}', f'{m}_change': f'{m} change'}
cols |= {'recovery_x': f'recovery used {dx} (%)', 'recovery_y': f'recovery used {dy} (%)'}   # the recovery behind each priced number (40% unless a wide name was marked down)
view = res[['member_name', 'Market_CDS_Ticker', *cols, 'status']].rename(columns=cols)
places = {name: (2 if name.startswith('risky_pv01') else 1) for name in cols.values()}   # risky_pv01: 2 decimals, everything else 1
show(view, numeric=places, flag_col='status')   # red = missing data, amber = recalibrated

median spread change 21.6 bp | median pct change 8.4%


member_name,Market_CDS_Ticker,spread 2026-09-29,spread 2026-10-26,spread change (bp),spread change (%),upfront 2026-09-29,upfront 2026-10-26,upfront change,all_upfront 2026-09-29,all_upfront 2026-10-26,all_upfront change,risky_pv01 2026-09-29,risky_pv01 2026-10-26,risky_pv01 change,recovery used 2026-09-29 (%),recovery used 2026-10-26 (%),status
Virgin Media Finance PLC,VMED,1586.3,5297.5,3711.2,234.0,28.6,66.6,38.0,43.1,74.9,31.8,2.90,1.67,-1.23,40.0,20.1,Y recal
Ziggo Bond Co BV,ZIGGO,594.4,1552.7,958.4,161.2,3.5,27.8,24.3,23.7,42.1,18.4,4.03,2.86,-1.17,40.0,40.0,ok
Motion Finco Sarl,MERLLN B,1763.3,2657.4,894.1,50.7,31.8,41.8,9.9,45.1,52.4,7.3,2.65,2.13,-0.52,40.0,40.0,X addin; Y addin
INEOS Quattro Finance 2 Plc,STYRO,1154.4,1770.5,616.1,53.4,19.9,31.3,11.4,36.2,43.8,7.6,3.25,2.49,-0.76,40.0,40.0,ok
Multiversity SpA,MLTIVR,555.8,873.8,318.0,57.2,2.1,12.5,10.4,22.8,30.0,7.2,4.14,3.51,-0.63,40.0,40.0,ok
Monitchem HoldCo 3 SA,CABBCO,697.8,961.2,263.4,37.7,7.1,14.9,7.8,26.7,32.4,5.7,3.92,3.50,-0.42,40.0,40.0,ok
Altice France SA,SFRFP,296.6,545.0,248.3,83.7,-8.6,1.7,10.3,13.2,21.4,8.2,4.36,3.94,-0.42,40.0,40.0,ok
United Group BV,ADRBID,334.1,477.7,143.6,43.0,-6.9,-0.9,6.0,15.1,19.9,4.9,4.39,4.16,-0.23,40.0,40.0,ok
Worldline SA/France,WLNFP,854.2,950.3,96.0,11.2,12.0,14.6,2.6,29.7,31.6,1.9,3.54,3.40,-0.14,40.0,40.0,ok
Fibercop SpA,FIBCOP,173.0,263.2,90.2,52.1,-14.5,-10.0,4.5,8.5,12.3,3.7,4.60,4.45,-0.15,40.0,40.0,ok


## 3. Pricing coverage, all indices
How much of the store is priced, per index (cells = index x member x date). `priced` is Databricks by the agreed definition; `quoted at index coupon %` counts cells Markit quotes at the index's own coupon (the rest are re-priced). `filled from add-in` is how many of the `not priced` cells the Markit add-in prices, `priced % incl. add-in` the resulting coverage, and `still not priced` what is left. `conv=0 (priced via par)` counts wide-name rows that Markit gives no conv_spread but which still count as priced.

In [5]:
show(cs.pricing_coverage(), numeric={'priced %': 1, 'quoted at index coupon %': 1, 'priced % incl. fills': 1})

index,cells,priced,priced %,quoted at index coupon %,not priced,filled from add-in,manual overrides,priced % incl. fills,still not priced,no row,empty row,conv=0 (priced via par)
IG47,3250,3172,97.6,83.2,78,78,0,100.0,0,78,0,0
IG46,19625,19354,98.6,85.0,271,271,0,100.0,0,271,0,0
IG45,35750,34917,97.7,83.3,833,794,0,99.9,39,832,1,0
IG44,52250,51285,98.2,84.6,965,794,0,99.7,171,964,1,0
IG43,68375,66384,97.1,83.5,1991,1691,0,99.6,300,1989,2,0
IG42,84875,80386,94.7,81.9,4489,4057,0,99.5,432,4488,1,0
IG41,101125,93367,92.3,80.3,7758,7195,0,99.4,563,7757,1,0
IG40,117625,109178,92.8,80.8,8447,7467,0,99.2,980,8447,0,0
IG39,133750,123755,92.5,81.3,9995,8400,0,98.8,1595,9995,0,0
IG38,149000,136678,91.7,82.9,12322,10097,0,98.5,2225,12321,1,0


## 4. Index metrics
Built from each name's `all_upfront` (its present value of expected loss, in points; Markit's value from Databricks where priced there, else the Markit add-in). One row per index and date.

- **dispersion**: standard deviation of `all_upfront` across the index's priced names (unweighted, population std).
- **index_el**: index expected loss = sum of weight x `all_upfront` over the priced names, divided by the weight of all names live on that date (points).
- **top5_share / top10_share**: expected loss of the 5 / 10 widest names (largest `all_upfront`) x their weights, divided by the total over priced names.

Names that defaulted out of a series (from `teams.structured_credit_abhutra.credit_events`) stay in it at their original weight until `index_ex_date`, using their own prices (Databricks, else the Markit add-in), so earlier dates still include them. Unpriced names are left out of that date; `n_priced` and `priced_weight` show how much of the live index each number covers.

In [6]:
import index_metrics as im

met = im.index_metrics(INDEX).set_index('published_date')
two = met.loc[[a['date_x'], a['date_y']], ['dispersion', 'index_el', 'top5_share', 'top10_share', 'max_el', 'n_priced', 'n_members', 'priced_weight']].copy()
two.loc['change'] = two.iloc[1] - two.iloc[0]
for c in ('top5_share', 'top10_share', 'priced_weight'):
    two[c] = two[c] * 100                       # show as %
two = two.rename(columns={'top5_share': 'top5 share (%)', 'top10_share': 'top10 share (%)', 'priced_weight': 'priced weight (%)', 'index_el': 'index EL (pts)', 'max_el': 'widest (pts)'})
two.insert(0, 'date', [f"{a['date_x']:%Y-%m-%d}", f"{a['date_y']:%Y-%m-%d}", 'change'])
show(two, numeric={'dispersion': 2, 'index EL (pts)': 2, 'top5 share (%)': 1, 'top10 share (%)': 1, 'widest (pts)': 1, 'n_priced': 0, 'n_members': 0, 'priced weight (%)': 1})

date,dispersion,index EL (pts),top5 share (%),top10 share (%),widest (pts),n_priced,n_members,priced weight (%)
2026-09-29,8.49,12.86,19.4,32.3,45.1,75,75,100.0
2026-10-26,11.57,14.68,22.3,35.0,75.1,75,75,100.0
change,3.09,1.82,2.9,2.6,30.0,0,0,0.0


The widest names on the end date, and what share of the index's expected loss each carries:

In [7]:
w = im.widest(INDEX, a['date_y'], 10)
w['weight'] = w['weight'] * 100
w['share_of_index_el'] = w['share_of_index_el'] * 100
w = w.rename(columns={'all_upfront': 'all_upfront (pts)', 'weight': 'weight (%)', 'el_contribution': 'EL contribution (pts)', 'share_of_index_el': 'share of index EL (%)'})
show(w, numeric={'all_upfront (pts)': 1, 'weight (%)': 2, 'EL contribution (pts)': 2, 'share of index EL (%)': 1})

member_name,market_cds_ticker,all_upfront (pts),weight (%),EL contribution (pts),share of index EL (%),source
Virgin Media Finance PLC,VMED,75.1,1.33,1.00,6.8,databricks
Motion Finco Sarl,MERLLN B,52.4,1.33,0.70,4.8,addin
INEOS Quattro Finance 2 Plc,STYRO,43.8,1.33,0.58,4.0,databricks
Ziggo Bond Co BV,ZIGGO,42.1,1.33,0.56,3.8,databricks
Monitchem HoldCo 3 SA,CABBCO,32.4,1.33,0.43,2.9,databricks
Worldline SA/France,WLNFP,31.6,1.33,0.42,2.9,databricks
Multiversity SpA,MLTIVR,30.0,1.33,0.40,2.7,databricks
ION Platform Finance SARL,IONPLA,26.1,1.33,0.35,2.4,databricks
INEOS Finance PLC,INEGRP,26.0,1.33,0.35,2.4,databricks
Samhallsbyggnadsbolaget i Nord,SBBBSS,25.4,1.33,0.34,2.3,databricks
